### TAR with $\ell_2$-Norm

In [ ]:
import cupy as np
# import numpy as np

def rsvd(A, k, p=0, q=0):
    m, n = A.shape
    Omega = np.random.randn(n, k + p)
    Y = A @ Omega
    for _ in range(q):
        Y = A @ (A.T @ Y)
    Q, _ = np.linalg.qr(Y)
    B = Q.T @ A
    U_tilde, S, Vt = np.linalg.svd(B, full_matrices=False)
    U = Q @ U_tilde
    return U[:, :k], S[:k], Vt[:k, :]

def ten2mat(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order = 'F')

def update_cg(var, r, q, Aq, rold):
    alpha = rold / np.inner(q, Aq)
    var = var + alpha * q
    r = r - alpha * Aq
    rnew = np.inner(r, r)
    q = r + (rnew / rold) * q
    return var, r, q, rnew

def fold_G(G, Rw, Ru, Rv):
    Gt = np.zeros((Rw, Ru, Rv))
    for r in range(Rv):
        Gt[:, :, r] = G[:, r * Ru : (r + 1) * Ru]
    return Gt

In [ ]:
def solve_mu_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu, dv -> uvt', X, U, V)
    return np.mean(Y[:, d :] - np.einsum('nw, wuv, uvt -> nt', W, fold_G(G, Rw, Ru, Rv), mat), axis = 1)

def solve_W_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k -1 : T - k -1]
    var0 = np.einsum('nu, ndt, dv -> uvt', U, X, V)
    var1 = np.einsum('wuv, uvt -> wt', fold_G(G, Rw, Ru, Rv), var0)
    return (Y[:, d :] - mu[:, np.newaxis]) @ np.linalg.pinv(var1)

def solve_G_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv):
    N, T = Y.shape
    X = np.zeros((N * d, T - d))
    for k in range(d):
        X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.kron(V, U).T @ X
    data = Y[:, d :] - mu[:, np.newaxis]
    temp = np.einsum('kt, ht -> kh', mat, mat)
    return np.linalg.inv(W.T @ W) @ W.T @ np.einsum('nt, kt -> nk', data, mat) @ np.linalg.inv(temp)

def ell_U_L2(var0, temp1, U):
    return np.einsum('njut, jt -> nu', var0,
                     np.einsum('jw, wt -> jt', temp1,
                               np.einsum('nwut, nu -> wt', var0, U)))

def solve_U_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, dv -> nvt', X, V)
    var0 = np.einsum('nvt, wuv -> nwut', mat, fold_G(G, Rw, Ru, Rv))
    data = Y[:, d :] - mu[:, np.newaxis]
    temp1 = W.T @ W
    u = np.reshape(U, -1, order = 'F')
    response = np.einsum('nwut, wt -> nu', var0, W.T @ data)
    r = np.reshape(response - ell_U_L2(var0, temp1, U), -1, order = 'F')
    q = r.copy()
    rold = np.inner(r, r)
    for it in range(5): # CG iteration
        Q = np.reshape(q, (N, Ru), order = 'F')
        Aq = np.reshape(ell_U_L2(var0, temp1, Q), -1, order = 'F')
        u, r, q, rold = update_cg(u, r, q, Aq, rold)
    U = np.reshape(u, (N, Ru), order = 'F')
    return U

def ell_V_L2(var0, temp1, V):
    return np.einsum('djvt, jt -> dv', var0,
                     np.einsum('jw, wt -> jt', temp1,
                               np.einsum('dwvt, dv -> wt', var0, V)))

def solve_V_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu -> dut', X, U)
    var0 = np.einsum('dut, wuv -> dwvt', mat, fold_G(G, Rw, Ru, Rv))
    data = Y[:, d :] - mu[:, np.newaxis]
    temp1 = W.T @ W
    v = np.reshape(V, -1, order = 'F')
    response = np.einsum('dwvt, wt -> dv', var0, W.T @ data)
    r = np.reshape(response - ell_V_L2(var0, temp1, V), -1, order = 'F')
    q = r.copy()
    rold = np.inner(r, r)
    for it in range(5): # CG iteration
        Q = np.reshape(q, (d, Rv), order = 'F')
        Aq = np.reshape(ell_V_L2(var0, temp1, Q), -1, order = 'F')
        v, r, q, rold = update_cg(v, r, q, Aq, rold)
    V = np.reshape(v, (d, Rv), order = 'F')
    return V

In [ ]:
import cupy as np
import time

def tar(Y, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    mu = np.zeros(N)
    factor, _, _ = rsvd(Y[:, d :], Rw)
    W = factor[:, : Rw]
    X = np.zeros((N, d * (T - d)))
    for k in range(d):
        X[:, k * (T - d) : (k + 1) * (T - d)] = Y[:, d - k - 1 : T - k - 1]
    factor, _, _ = rsvd(X, Ru)
    U = factor[:, : Ru]
    X = np.zeros((d, N * (T - d)))
    for k in range(d):
        X[k, :] = Y[:, d - k - 1 : T - k - 1].reshape(-1, order = 'F')
    factor, _, _ = rsvd(X, Rv)
    V = factor[:, : Rv]
    X = np.zeros((N * d, T - d))
    for k in range(d):
        X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
    G = np.linalg.pinv(W) @ Y[:, d :] @ np.linalg.pinv((np.kron(V, U)).T @ X)
    for iter in range(max_iter):
        mu = solve_mu_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv)
        W = solve_W_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv)
        G = solve_G_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv)
        U = solve_U_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv)
        V = solve_V_L2(Y, mu, W, G, U, V, d, Rw, Ru, Rv)
    return mu, W, G, U, V

In [ ]:
# import numpy as np

def compute_mape(var, var_hat):
    return np.sum(np.abs(var - var_hat) / var) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

def forecast(series, mu, W, G, U, V, horizon):
    N, t = series.shape
    d, Rv = V.shape
    Rw = W.shape[1]
    Ru = U.shape[1]
    est = np.zeros((N, horizon))
    for h in range(horizon):
        Xt = np.zeros((N, d))
        for k in range(d):
            Xt[:, k] = series[:, t - horizon + h - (k + 1)]
        est[:, h] = mu + np.einsum('nw, w -> n', W,
                                   np.einsum('wuv, uv -> w',
                                             fold_G(G, Rw, Ru, Rv), U.T @ Xt @ V))
    return est

In [ ]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

horizon = 168
d = 7 * 24
for R in [3]:
    print('Rank = {}'.format(R))
    start = time.time()
    np.random.seed(0)
    mu, W, G, U, V = tar(np.array(data[:, : 17 * 24]), d, R, R, R)
    end = time.time()
    print('Running time: %d seconds'%(end - start))

    ground = np.array(data[:, 24 * 17 : 24 * 24])
    est = forecast(np.array(data[:, : 24 * 24]), mu, W, G, U, V, horizon)
    pos = np.where(ground > 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

In [ ]:
import cupy as np

horizon = 168

ground = np.array(data[:, - horizon :])
est = forecast(np.array(data), mu, W, G, U, V, horizon)
pos = np.where(ground > 0)
print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
print()

### TAR with Robust Losses

In [ ]:
import cupy as np

def update_cg(var, r, q, Aq, rold):
    alpha = rold / np.inner(q, Aq)
    var = var + alpha * q
    r = r - alpha * Aq
    rnew = np.inner(r, r)
    q = r + (rnew / rold) * q
    return var, r, q, rnew

def fold_G(G, Rw, Ru, Rv):
    Gt = np.zeros((Rw, Ru, Rv))
    for r in range(Rv):
        Gt[:, :, r] = G[:, r * Ru : (r + 1) * Ru]
    return Gt

In [ ]:
def solve_mu(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu, dv -> uvt', X, U, V)
    C = Y[:, d :] - W @ np.einsum('wuv, uvt -> wt', fold_G(G, Rw, Ru, Rv), mat)
    for iter in range(max_iter):
        epsilon = C - mu[:, np.newaxis]
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        p = 2/3
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################
        mu = np.einsum('nt, nt -> n', theta, C) / np.sum(theta, axis = 1)
    return mu

In [ ]:
def ell_W(temp, W):
    return np.einsum('nwk, nw -> nk', temp, W).reshape(-1, order = 'F')

def solve_W(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    var0 = np.einsum('nu, ndt, dv -> uvt', U, X, V)
    var1 = np.einsum('wuv, uvt -> wt', fold_G(G, Rw, Ru, Rv), var0)
    temp1 = np.einsum('wt, kt -> wkt', var1, var1)
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
        epsilon = data - np.einsum('nw, wt -> nt', W, var1)
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        p = 2/3
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        w = np.reshape(W, -1, order = 'F')
        response = np.einsum('nt, wt -> nw', theta * data, var1).reshape(-1, order = 'F')
        temp = np.einsum('nt, wkt -> nwk', theta, temp1)
        r = response - ell_W(temp, W)
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (N, Rw), order = 'F')
            Aq = ell_W(temp, Q)
            w, r, q, rold = update_cg(w, r, q, Aq, rold)
        W = np.reshape(w, (N, Rw), order = 'F')
    return W

In [ ]:
def ell_G(mat, temp, G, Rw, Ru, Rv):
    temp1 = np.einsum('wuv, uvt -> wt', fold_G(G, Rw, Ru, Rv), mat)
    return np.einsum('lt, uvt -> luv', np.einsum('lwt, wt -> lt', temp, temp1), mat).reshape(-1, order = 'F')

def solve_G(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu, dv -> uvt', X, U, V)
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
        epsilon = data - np.einsum('nw, wuv, uvt -> nt', W, fold_G(G, Rw, Ru, Rv), mat)
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        p = 2/3
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        temp = np.einsum('ij, it, ik -> jkt', W, theta, W)

        ### Conjugate Gradient ###
        g = np.reshape(G, -1, order = 'F')
        response = np.einsum('wt, uvt -> wuv', W.T @ (theta * data), mat).reshape(-1, order = 'F')
        r = response - ell_G(mat, temp, G, Rw, Ru, Rv)
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (Rw, int(Ru * Rv)), order = 'F')
            Aq = ell_G(mat, temp, Q, Rw, Ru, Rv)
            g, r, q, rold = update_cg(g, r, q, Aq, rold)
        G = np.reshape(g, (Rw, int(Ru * Rv)), order = 'F')
    return G

In [ ]:
def ell_U(var0, temp1, U):
    return np.einsum('njut, jt -> nu', var0, np.einsum('jwt, wt -> jt', temp1, np.einsum('nwut, nu -> wt', var0, U)))

def solve_U(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, dv -> nvt', X, V)
    var0 = np.einsum('nvt, wuv -> nwut', mat, fold_G(G, Rw, Ru, Rv))
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
        epsilon = data - np.einsum('nw, wt -> nt', W, np.einsum('nwut, nu -> wt', var0, U))
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        p = 2/3
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        temp1 = np.einsum('ij, it, ik -> jkt', W, theta, W)

        ### Conjugate Gradient ###
        u = np.reshape(U, -1, order = 'F')
        response = np.einsum('nwut, wt -> nu', var0, W.T @ (theta * data))
        r = np.reshape(response - ell_U(var0, temp1, U), -1, order = 'F')
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (N, Ru), order = 'F')
            Aq = np.reshape(ell_U(var0, temp1, Q), -1, order = 'F')
            u, r, q, rold = update_cg(u, r, q, Aq, rold)
        U = np.reshape(u, (N, Ru), order = 'F')
    return U

In [ ]:
def ell_V(var0, temp1, V):
    return np.einsum('djvt, jt -> dv', var0, np.einsum('jwt, wt -> jt', temp1, np.einsum('dwvt, dv -> wt', var0, V)))

def solve_V(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu -> dut', X, U)
    var0 = np.einsum('dut, wuv -> dwvt', mat, fold_G(G, Rw, Ru, Rv))
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
        epsilon = data - np.einsum('nw, wt -> nt', W, np.einsum('dwvt, dv -> wt', var0, V))
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        p = 2/3
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        temp1 = np.einsum('ij, it, ik -> jkt', W, theta, W)

        ### Conjugate Gradient ###
        v = np.reshape(V, -1, order = 'F')
        response = np.einsum('dwvt, wt -> dv', var0, W.T @ (theta * data))
        r = np.reshape(response - ell_V(var0, temp1, V), -1, order = 'F')
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (d, Rv), order = 'F')
            Aq = np.reshape(ell_V(var0, temp1, Q), -1, order = 'F')
            v, r, q, rold = update_cg(v, r, q, Aq, rold)
        V = np.reshape(v, (d, Rv), order = 'F')
    return V

In [ ]:
# import cupy as np
import time

def huber_tar(Y, delta, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    # mu, W, G, U, V = tar(Y, d, R, R, R)
    ##############
    mu = np.zeros(N)
    factor, _, _ = rsvd(Y[:, d :], Rw)
    W = factor[:, : Rw]
    X = np.zeros((N, d * (T - d)))
    for k in range(d):
        X[:, k * (T - d) : (k + 1) * (T - d)] = Y[:, d - k - 1 : T - k - 1]
    factor, _, _ = rsvd(X, Ru)
    U = factor[:, : Ru]
    X = np.zeros((d, N * (T - d)))
    for k in range(d):
        X[k, :] = Y[:, d - k - 1 : T - k - 1].reshape(-1, order = 'F')
    factor, _, _ = rsvd(X, Rv)
    V = factor[:, : Rv]
    X = np.zeros((N * d, T - d))
    for k in range(d):
        X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
    G = np.linalg.pinv(W) @ Y[:, d :] @ np.linalg.pinv((np.kron(V, U)).T @ X)
    ###############
    for iter in range(max_iter):
        mu = solve_mu(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv)
        W = solve_W(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv)
        G = solve_G(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv)
        U = solve_U(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv)
        V = solve_V(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv)
    return mu, W, G, U, V

In [ ]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

horizon = 168
d = 7 * 24
for R in [3]:
    for delta in [40]:
        print('Rank = {}'.format(R))
        print('Delta = {}'.format(delta))
        start = time.time()
        np.random.seed(0)
        mu, W, G, U, V = huber_tar(np.array(data[:, : 17 * 24]), delta, d, R, R, R, 50)
        end = time.time()
        print('Running time: %d seconds'%(end - start))

        ground = np.array(data[:, 24 * 17 : 24 * 24])
        est = forecast(np.array(data[:, : 24 * 24]), mu, W, G, U, V, horizon)
        pos = np.where(ground > 0)
        print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
        print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
        print()

In [ ]:
ground = np.array(data[:, - horizon :])
est = forecast(np.array(data), mu, W, G, U, V, horizon)
pos = np.where(ground > 0)
print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
print()